In [7]:
import numpy as np
from PIL import Image
from gym import Wrapper
from gym.spaces import Box
from collections import deque

import gym_super_mario_bros
from nes_py.wrappers import JoypadSpace


# -----------------------------
# Skip 4 frames
# -----------------------------

class SkipFrame(Wrapper):

    def __init__(self, env, skip=4):
        super().__init__(env)
        self.skip = skip

    def step(self, action):

        total_reward = 0

        for _ in range(self.skip):

            state, reward, terminated, truncated, info = self.env.step(action)

            total_reward += reward

            if terminated or truncated:
                break

        return state, total_reward, terminated, truncated, info


# -----------------------------
# Grayscale
# -----------------------------

class GrayScale(Wrapper):

    def __init__(self, env):
        super().__init__(env)

        self.observation_space = Box(
            low=0,
            high=255,
            shape=(240, 256, 1),
            dtype=np.uint8
        )

    def reset(self):
        state, info = self.env.reset()

        state = np.mean(state, axis=2).astype(np.uint8)
        state = state[:, :, None]

        return state, info

    def step(self, action):

        state, reward, terminated, truncated, info = self.env.step(action)

        state = np.mean(state, axis=2).astype(np.uint8)
        state = state[:, :, None]

        return state, reward, terminated, truncated, info


# -----------------------------
# Resize to 84 × 84
# -----------------------------

class Resize(Wrapper):

    def __init__(self, env):
        super().__init__(env)

        self.observation_space = Box(
            low=0,
            high=255,
            shape=(84, 84, 1),
            dtype=np.uint8
        )

    def resize(self, state):

        state = Image.fromarray(state.squeeze())
        state = state.resize((84, 84))

        return np.array(state)[:, :, None]

    def reset(self):

        state, info = self.env.reset()

        return self.resize(state), info

    def step(self, action):

        state, reward, terminated, truncated, info = self.env.step(action)

        return (
            self.resize(state),
            reward,
            terminated,
            truncated,
            info
        )


# -----------------------------
# Stack 4 frames
# -----------------------------

class FrameStack(Wrapper):

    def __init__(self, env, num_frames=4):
        super().__init__(env)

        self.frames = deque(maxlen=num_frames)

        self.observation_space = Box(
            low=0,
            high=255,
            shape=(num_frames, 84, 84),
            dtype=np.uint8
        )

    def reset(self):

        state, info = self.env.reset()

        self.frames.clear()

        for _ in range(4):
            self.frames.append(state)

        return np.array(self.frames).squeeze(-1), info

    def step(self, action):

        state, reward, terminated, truncated, info = self.env.step(action)

        self.frames.append(state)

        return (
            np.array(self.frames).squeeze(-1),
            reward,
            terminated,
            truncated,
            info
        )


# ============================================
# Create Mario
# ============================================

env = gym_super_mario_bros.make(
    "SuperMarioBros-1-1-v0"
)


# ============================================
# Our 3 actions
# ============================================

env = JoypadSpace(
    env,
    [
        ["right"],
        ["right", "A"],
        ["right", "B"]
    ]
)


# ============================================
# Apply preprocessing
# ============================================

env = SkipFrame(env)
env = GrayScale(env)
env = Resize(env)
env = FrameStack(env)


# ============================================
# Test
# ============================================

state, info = env.reset()

print("Action space:", env.action_space)
print("State shape:", state.shape)
print("State dtype:", state.dtype)


action = 0

next_state, reward, terminated, truncated, info = env.step(action)

print("Next state shape:", next_state.shape)
print("Reward:", reward)
print("Done:", terminated or truncated)


env.close()

Action space: Discrete(3)
State shape: (4, 84, 84)
State dtype: uint8
Next state shape: (4, 84, 84)
Reward: 0.0
Done: False


In [3]:
              #       AGENT
              #         │
              #         │ action
              #         ▼
              #  ┌──────────────┐
              #  │ Mario Env    │
              #  └──────────────┘
              #         │
              #         │ raw image
              #         ▼
              #   240×256×3
              #         │
              #         ▼
              #    SkipFrame
              #         │
              #         ▼
              #     Grayscale
              #         │
              #         ▼
              #    Resize 84×84
              #         │
              #         ▼
              #   FrameStack ×4
              #         │
              #         ▼
              #    4×84×84
              #         │
              #         ▼
              #     DQN CNN
              #         │
              #         ▼
              # Q-values for 3 actions

              # ┌─────────────────┐
              # │ RIGHT            │
              # │ RIGHT + A        │
              # │ RIGHT + B        │
              # └─────────────────┘

IndentationError: unindent does not match any outer indentation level (<string>, line 5)